In [5]:
# Create the docker-compose.yml file programmatically
docker_compose_content = """
services:
  neo4j:
    image: neo4j:latest
    container_name: neo4j_banking_assignment
    ports:
      - "7474:7474" # HTTP
      - "7687:7687" # Bolt (Driver Connection)
    environment:
      - NEO4J_AUTH=neo4j/password
      - NEO4J_PLUGINS=["apoc", "graph-data-science"]
      - NEO4J_dbms_security_procedures_unrestricted=apoc.*,gds.*
      - NEO4J_dbms_security_procedures_allowlist=apoc.*,gds.*
      - NEO4J_dbms_memory_heap_max__size=2G
    volumes:
      - ./neo4j/data:/data
      - ./neo4j/import:/import
      - ./neo4j/plugins:/plugins
      - ./neo4j/logs:/logs
"""

with open("docker-compose.yml", "w") as f:
    f.write(docker_compose_content.strip())

print("✅ docker-compose.yml created successfully.")

✅ docker-compose.yml created successfully.


In [6]:
# Start the container in detached mode
!docker compose up -d

print("🚀 Container is starting up. Please wait about 30-45 seconds for plugins to initialize...")

[+] up 1/1
 ✔ Container neo4j_banking_assignment Running                               0.0s
🚀 Container is starting up. Please wait about 30-45 seconds for plugins to initialize...


In [7]:
%pip install requests pandas neo4j


[notice] A new release of pip is available: 23.0 -> 26.0.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import time
from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")

def verify_setup():
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        # Give the container a few retries in case it's still booting
        for i in range(10):
            try:
                driver.verify_connectivity()
                with driver.session() as session:
                    gds_v = session.run("RETURN gds.version() AS v").single()["v"]
                    apoc_v = session.run("RETURN apoc.version() AS v").single()["v"]
                    return f"✅ Connected! GDS: {gds_v}, APOC: {apoc_v}"
            except Exception:
                print(f"Waiting for Neo4j... (Attempt {i+1}/10)")
                time.sleep(10)
        return "❌ Connection failed. Check Docker logs."

print(verify_setup())

✅ Connected! GDS: 2.27.0, APOC: 2026.02.2


In [9]:
import os

# Create the import directory if it doesn't exist
os.makedirs("./neo4j/import", exist_ok=True)
print("✅ Directory ./neo4j/import is ready.")

✅ Directory ./neo4j/import is ready.


In [10]:
import requests
import os

def download_gist_files(gist_id):
    # Call the GitHub API to get gist metadata
    api_url = f"https://api.github.com/gists/{gist_id}"
    response = requests.get(api_url)
    response.raise_for_status()
    gist_data = response.json()
    
    import_dir = "./neo4j/import"
    os.makedirs(import_dir, exist_ok=True)
    
    print(f"Found {len(gist_data['files'])} files in Gist...")
    
    for file_info in gist_data['files'].values():
        filename = file_info['filename']
        raw_url = file_info['raw_url']
        
        # Standardize naming for your assignment: 'account.csv' -> 'accounts.csv'
        local_filename = "accounts.csv" if filename == "account.csv" else filename
        
        try:
            print(f"Downloading {filename}...")
            file_content = requests.get(raw_url).content
            with open(os.path.join(import_dir, local_filename), "wb") as f:
                f.write(file_content)
            print(f"✅ Saved as: {local_filename}")
        except Exception as e:
            print(f"❌ Failed {filename}: {e}")

# Use the ID from your link
download_gist_files("19bf6eb6f6a5adecddedf2a081b51789")

Found 3 files in Gist...
✅ Saved as: customers.csv
✅ Saved as: purchases.csv
✅ Saved as: transfers.csv


In [11]:
# Run this AFTER the container is up and verify_setup() has passed
def check_import_folder():
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:
            # List files in the import directory using APOC
            result = session.run("CALL apoc.import.csv([], [], {file: ''}) YIELD file RETURN file")
            # Note: A simpler check is just calling a file list via system
            print("Files available to Neo4j:")
            !docker exec neo4j_banking_assignment ls /import

check_import_folder()

Files available to Neo4j:
customers.csv
purchases.csv
transfers.csv
